# PostgresFileSystem Playground

This notebook creates a small PostgreSQL-backed playground for `PostgresFileSystem` so you can see what the backend stores and how the native search paths behave.

What it covers:

1. Provision a local playground database and the required Postgres indexes.
2. Seed a small corpus with native pgvector embeddings.
3. Inspect the raw `vfs_objects` rows and the indexes that `verify_native_search_schema()` checks.
4. Exercise `glob`, `grep`, `lexical_search`, `vector_search`, `semantic_search`, and the CLI query surface.


## Prereqs

- Local PostgreSQL running and reachable from this machine.
- The optional Postgres dependencies installed, for example `uv sync --extra postgres`.
- Run the notebook from the repo root or from `examples/`.
- If your local Postgres URL is different, set `VFS_POSTGRES_NOTEBOOK_URL` before launching Jupyter, or edit `DemoConfig` below.

The default playground URL is `postgresql+asyncpg://localhost/vfs_postgres_playground`.


In [1]:
from __future__ import annotations

import html
import math
import os
import re
import sys
from dataclasses import asdict, dataclass
from pathlib import Path

from IPython.display import HTML, Markdown, display
from sqlalchemy import text
from sqlalchemy.engine import make_url
from sqlalchemy.ext.asyncio import AsyncEngine, create_async_engine

repo_candidates = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((path.resolve() for path in repo_candidates if (path / "pyproject.toml").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repo root or from examples/")
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from vfs.backends import PostgresFileSystem
from vfs.models import postgres_native_vfs_object_model, postgres_vector_column_spec
from vfs.vector import Vector


@dataclass(frozen=True)
class DemoConfig:
    database_url: str = os.getenv(
        "VFS_POSTGRES_NOTEBOOK_URL",
        "postgresql+asyncpg://localhost/vfs_postgres_playground",
    )
    reset_table: bool = True
    embedding_dimension: int = 4


AXES = ("auth", "billing", "postgres", "python")
TOKEN_TO_AXIS = {
    "auth": 0,
    "authenticate": 0,
    "authentication": 0,
    "login": 0,
    "password": 0,
    "reset": 0,
    "session": 0,
    "sessions": 0,
    "signin": 0,
    "token": 0,
    "tokens": 0,
    "user": 0,
    "users": 0,
    "billing": 1,
    "card": 1,
    "cards": 1,
    "charge": 1,
    "charges": 1,
    "invoice": 1,
    "invoices": 1,
    "payment": 1,
    "payments": 1,
    "refund": 1,
    "backend": 2,
    "fts": 2,
    "gin": 2,
    "glob": 2,
    "grep": 2,
    "index": 2,
    "indexes": 2,
    "pgvector": 2,
    "postgres": 2,
    "postgresql": 2,
    "schema": 2,
    "search": 2,
    "vector": 2,
    "async": 3,
    "await": 3,
    "class": 3,
    "function": 3,
    "module": 3,
    "py": 3,
    "pytest": 3,
    "python": 3,
    "service": 3,
}
DEMO_VECTOR = Vector[4]

DEMO_FILES = [
    {
        "path": "/playbooks/auth/login.md",
        "content": """# Login flow\n\nUsers sign in with email plus a one-time code. Session tokens expire after 15 minutes. The auth service stores audit events in Postgres so lexical search can find login regressions quickly.\n""",
    },
    {
        "path": "/playbooks/auth/password-reset.md",
        "content": """# Password reset\n\nPassword reset invalidates old sessions, issues a fresh token, and records the reset event for auth debugging.\n""",
    },
    {
        "path": "/playbooks/billing/invoices.md",
        "content": """# Invoice retries\n\nBilling retries failed card charges, regenerates invoice PDFs, and sends payment recovery emails when a charge is declined.\n""",
    },
    {
        "path": "/docs/postgres_backend.md",
        "content": """# Postgres backend\n\nPostgresFileSystem keeps grep, glob, lexical search, and pgvector similarity inside PostgreSQL. verify_native_search_schema checks the GIN full-text index and the ANN vector index before traffic hits the backend.\n""",
    },
    {
        "path": "/src/auth_service.py",
        "content": """async def validate_token(token: str) -> bool:\n    if token == \"expired\":\n        return False\n    return True\n\n\nasync def login_user(email: str) -> str:\n    session_token = f\"session:{email}\"\n    return session_token\n""",
    },
    {
        "path": "/src/billing_service.py",
        "content": """def retry_invoice_charge(invoice_id: str) -> str:\n    return f\"retrying card charge for {invoice_id}\"\n""",
    },
]


def toy_embed_text(text_value: str) -> Vector:
    counts = [0.0] * len(AXES)
    for token in re.findall(r"[a-z0-9_]+", text_value.lower()):
        axis = TOKEN_TO_AXIS.get(token)
        if axis is not None:
            counts[axis] += 1.0
    if not any(counts):
        counts = [1.0] * len(AXES)
    norm = math.sqrt(sum(value * value for value in counts))
    return DEMO_VECTOR([round(value / norm, 4) for value in counts])


class ToyEmbeddingProvider:
    async def embed(self, text_value: str) -> Vector:
        return toy_embed_text(text_value)

    async def embed_batch(self, texts: list[str]) -> list[Vector]:
        return [toy_embed_text(text_value) for text_value in texts]

    @property
    def dimensions(self) -> int:
        return len(AXES)

    @property
    def model_name(self) -> str:
        return "toy-postgres-demo"


def _format_value(value: object) -> str:
    if value is None:
        return ""
    if isinstance(value, float):
        return f"{value:.4f}"
    if isinstance(value, (list, tuple)):
        return ", ".join(_format_value(item) for item in value)
    return str(value)


def _preview(text_value: str | None, limit: int = 96) -> str:
    if not text_value:
        return ""
    compact = " ".join(text_value.split())
    return compact if len(compact) <= limit else compact[: limit - 3] + "..."


def show_table(rows: list[dict[str, object]], *, title: str | None = None) -> None:
    if title:
        display(Markdown(f"**{title}**"))
    if not rows:
        print("(no rows)")
        return

    columns = list(rows[0].keys())
    html_parts = [
        "<table>",
        "<thead><tr>",
        *(f"<th style='text-align:left;padding:6px'>{html.escape(str(column))}</th>" for column in columns),
        "</tr></thead><tbody>",
    ]
    for row in rows:
        html_parts.append("<tr>")
        for column in columns:
            cell = html.escape(_format_value(row.get(column)))
            html_parts.append(
                "<td style='vertical-align:top;padding:6px'><pre style='margin:0;white-space:pre-wrap'>"
                + cell
                + "</pre></td>"
            )
        html_parts.append("</tr>")
    html_parts.append("</tbody></table>")
    display(HTML("".join(html_parts)))


def result_rows(result, columns: tuple[str, ...]) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    for entry in result.entries:
        row: dict[str, object] = {}
        for column in columns:
            if column == "preview":
                row[column] = _preview(entry.content)
            elif column == "matches":
                row[column] = ", ".join(str(line.match) for line in (entry.lines or []))
            else:
                row[column] = getattr(entry, column, None)
        rows.append(row)
    return rows


def show_result(result, *, columns: tuple[str, ...]) -> None:
    if not result.success:
        show_table([{"error": error} for error in result.errors], title=f"{result.function} errors")
        return
    show_table(result_rows(result, columns), title=f"{result.function} ({len(result.entries)} hits)")


async def sql_rows(engine: AsyncEngine, sql: str, params: dict[str, object] | None = None) -> list[dict[str, object]]:
    async with engine.connect() as conn:
        result = await conn.execute(text(sql), params or {})
        return [dict(row._mapping) for row in result]


async def ensure_database(database_url: str) -> str:
    url = make_url(database_url)
    database_name = url.database or ""
    if re.fullmatch(r"[a-z][a-z0-9_]*", database_name) is None:
        raise ValueError(
            "For safety, use a simple database name like 'vfs_postgres_playground' in VFS_POSTGRES_NOTEBOOK_URL"
        )
    admin_engine = create_async_engine(url.set(database="postgres"), echo=False)
    try:
        async with admin_engine.connect() as raw_conn:
            conn = await raw_conn.execution_options(isolation_level="AUTOCOMMIT")
            exists = (
                await conn.execute(
                    text("SELECT 1 FROM pg_database WHERE datname = :database_name"),
                    {"database_name": database_name},
                )
            ).scalar()
            if exists is None:
                await conn.execute(text(f"CREATE DATABASE {database_name}"))
    finally:
        await admin_engine.dispose()
    return url.render_as_string(hide_password=False)


async def build_playground(config: DemoConfig):
    if config.embedding_dimension != len(AXES):
        raise ValueError(
            f"This notebook's toy embedding model uses {len(AXES)} dimensions; keep embedding_dimension={len(AXES)} or update AXES and toy_embed_text()."
        )

    database_url = await ensure_database(config.database_url)
    model = postgres_native_vfs_object_model(dimension=config.embedding_dimension)
    vector_spec = postgres_vector_column_spec(model)
    engine = create_async_engine(database_url, echo=False)

    async with engine.begin() as conn:
        await conn.execute(text("CREATE EXTENSION IF NOT EXISTS vector"))
        if config.reset_table:
            await conn.execute(text(f"DROP TABLE IF EXISTS {model.__tablename__} CASCADE"))
        await conn.run_sync(model.metadata.create_all)
        await conn.execute(
            text(
                f"""
                CREATE INDEX IF NOT EXISTS ix_{model.__tablename__}_content_tsv_gin
                ON {model.__tablename__} USING GIN (to_tsvector('simple', coalesce(content, '')))
                """
            )
        )
        await conn.execute(
            text(
                f"""
                CREATE INDEX IF NOT EXISTS {vector_spec.index_name}
                ON {model.__tablename__} USING {vector_spec.index_method}
                ({vector_spec.column_name} {vector_spec.operator_class})
                WHERE {vector_spec.column_name} IS NOT NULL
                """
            )
        )

    fs = PostgresFileSystem(
        engine=engine,
        model=model,
        embedding_provider=ToyEmbeddingProvider(),
    )
    await fs.verify_native_search_schema()
    return engine, model, vector_spec, fs


async def seed_demo(fs: PostgresFileSystem, model):
    objects = [
        model(
            path=item["path"],
            content=item["content"],
            embedding=toy_embed_text(item["content"]),
        )
        for item in DEMO_FILES
    ]
    result = await fs.write(objects=objects)
    if not result.success:
        raise RuntimeError("\n".join(result.errors))
    return [
        {
            "path": item["path"],
            "embedding": list(toy_embed_text(item["content"])),
            "preview": _preview(item["content"]),
        }
        for item in DEMO_FILES
    ]


In [2]:
CONFIG = DemoConfig()

show_table([asdict(CONFIG)], title="Notebook config")
show_table(
    [
        {
            "axis": axis,
            "example keywords": ", ".join(sorted(token for token, idx in TOKEN_TO_AXIS.items() if idx == axis_index)[:8]),
        }
        for axis_index, axis in enumerate(AXES)
    ],
    title="Toy embedding axes",
)


**Notebook config**

database_url,reset_table,embedding_dimension
postgresql+asyncpg://localhost/vfs_postgres_playground,True,4


**Toy embedding axes**

axis,example keywords
auth,"auth, authenticate, authentication, login, password, reset, session, sessions"
billing,"billing, card, cards, charge, charges, invoice, invoices, payment"
postgres,"backend, fts, gin, glob, grep, index, indexes, pgvector"
python,"async, await, class, function, module, py, pytest, python"


In [3]:
if "engine" in globals():
    await engine.dispose()

engine, model, vector_spec, fs = await build_playground(CONFIG)

show_table(
    [
        {
            "database_url": CONFIG.database_url,
            "table": model.__tablename__,
            "vector_index": vector_spec.index_name,
            "vector_metric": vector_spec.operator_class,
        }
    ],
    title="Provisioned playground",
)


**Provisioned playground**

database_url,table,vector_index,vector_metric
postgresql+asyncpg://localhost/vfs_postgres_playground,vfs_objects,ix_vfs_objects_embedding_vector_cosine_hnsw,vector_cosine_ops


In [4]:
seeded_files = await seed_demo(fs, model)
show_table(seeded_files, title="Seeded files + embeddings")


**Seeded files + embeddings**

path,embedding,preview
/playbooks/auth/login.md,"0.9370, 0.0000, 0.3123, 0.1562",# Login flow Users sign in with email plus a one-time code. Session tokens expire after 15 mi...
/playbooks/auth/password-reset.md,"1.0000, 0.0000, 0.0000, 0.0000","# Password reset Password reset invalidates old sessions, issues a fresh token, and records t..."
/playbooks/billing/invoices.md,"0.0000, 1.0000, 0.0000, 0.0000","# Invoice retries Billing retries failed card charges, regenerates invoice PDFs, and sends pa..."
/docs/postgres_backend.md,"0.0000, 0.0000, 1.0000, 0.0000","# Postgres backend PostgresFileSystem keeps grep, glob, lexical search, and pgvector similari..."
/src/auth_service.py,"0.8321, 0.0000, 0.0000, 0.5547","async def validate_token(token: str) -> bool: if token == ""expired"": return False return True..."
/src/billing_service.py,"0.0000, 1.0000, 0.0000, 0.0000","def retry_invoice_charge(invoice_id: str) -> str: return f""retrying card charge for {invoice_..."


In [5]:
index_rows = await sql_rows(
    engine,
    """
    SELECT indexname, indexdef
    FROM pg_indexes
    WHERE schemaname = 'public' AND tablename = 'vfs_objects'
    ORDER BY indexname
    """,
)
show_table(index_rows, title="Indexes that PostgresFileSystem verifies")

object_rows = await sql_rows(
    engine,
    """
    SELECT
        path,
        kind,
        ext,
        size_bytes,
        left(coalesce(content, ''), 100) AS preview,
        CASE WHEN embedding IS NULL THEN NULL ELSE embedding::text END AS embedding
    FROM vfs_objects
    WHERE deleted_at IS NULL
    ORDER BY path
    """,
)
show_table(object_rows, title="Rows currently stored in vfs_objects")

print(await fs.cli("tree / --depth 3"))


**Indexes that PostgresFileSystem verifies**

indexname,indexdef
ix_vfs_objects_content_tsv_gin,"CREATE INDEX ix_vfs_objects_content_tsv_gin ON public.vfs_objects USING gin (to_tsvector('simple'::regconfig, (COALESCE(content, ''::character varying))::text))"
ix_vfs_objects_embedding_vector_cosine_hnsw,CREATE INDEX ix_vfs_objects_embedding_vector_cosine_hnsw ON public.vfs_objects USING hnsw (embedding vector_cosine_ops) WHERE (embedding IS NOT NULL)
ix_vfs_objects_ext,CREATE INDEX ix_vfs_objects_ext ON public.vfs_objects USING btree (ext)
ix_vfs_objects_ext_kind,"CREATE INDEX ix_vfs_objects_ext_kind ON public.vfs_objects USING btree (ext, kind)"
ix_vfs_objects_kind,CREATE INDEX ix_vfs_objects_kind ON public.vfs_objects USING btree (kind)
ix_vfs_objects_owner_id,CREATE INDEX ix_vfs_objects_owner_id ON public.vfs_objects USING btree (owner_id)
ix_vfs_objects_parent_path,CREATE INDEX ix_vfs_objects_parent_path ON public.vfs_objects USING btree (parent_path)
ix_vfs_objects_path,CREATE UNIQUE INDEX ix_vfs_objects_path ON public.vfs_objects USING btree (path)
ix_vfs_objects_source_path,CREATE INDEX ix_vfs_objects_source_path ON public.vfs_objects USING btree (source_path)
ix_vfs_objects_target_path,CREATE INDEX ix_vfs_objects_target_path ON public.vfs_objects USING btree (target_path)


**Rows currently stored in vfs_objects**

path,kind,ext,size_bytes,preview,embedding
/docs,directory,,0,,
/docs/postgres_backend.md,file,md,233,"# Postgres backend PostgresFileSystem keeps grep, glob, lexical search, and pgvector similarity ins","[0,0,1,0]"
/.vfs/docs/postgres_backend.md/__meta__/versions/1,version,,233,"# Postgres backend PostgresFileSystem keeps grep, glob, lexical search, and pgvector similarity ins",
/playbooks,directory,,0,,
/playbooks/auth,directory,,0,,
/playbooks/auth/login.md,file,md,204,# Login flow Users sign in with email plus a one-time code. Session tokens expire after 15 minutes.,"[0.937,0,0.3123,0.1562]"
/.vfs/playbooks/auth/login.md/__meta__/versions/1,version,,204,# Login flow Users sign in with email plus a one-time code. Session tokens expire after 15 minutes.,
/playbooks/auth/password-reset.md,file,md,129,"# Password reset Password reset invalidates old sessions, issues a fresh token, and records the res","[1,0,0,0]"
/.vfs/playbooks/auth/password-reset.md/__meta__/versions/1,version,,129,"# Password reset Password reset invalidates old sessions, issues a fresh token, and records the res",
/playbooks/billing,directory,,0,,


├── docs
│   └── postgres_backend.md
├── playbooks
│   ├── auth
│   │   ├── login.md
│   │   └── password-reset.md
│   └── billing
│       └── invoices.md
└── src
    ├── auth_service.py
    └── billing_service.py


In [6]:
glob_result = await fs.glob("/playbooks/**/*.md")
show_result(glob_result, columns=("path", "kind", "size_bytes"))

grep_result = await fs.grep(
    "token|invoice",
    paths=("/playbooks", "/src", "/docs"),
    case_mode="insensitive",
    columns=frozenset({"path", "content", "size_bytes"}),
)
show_result(grep_result, columns=("path", "score", "matches", "preview"))

lexical_result = await fs.lexical_search("postgres index token search", k=4)
show_result(lexical_result, columns=("path", "score", "preview"))


**glob (3 hits)**

path,kind,size_bytes
/playbooks/auth/login.md,file,204
/playbooks/auth/password-reset.md,file,129
/playbooks/billing/invoices.md,file,143


**grep (5 hits)**

path,score,matches,preview
/playbooks/auth/login.md,1.0000,3,# Login flow Users sign in with email plus a one-time code. Session tokens expire after 15 mi...
/playbooks/auth/password-reset.md,1.0000,3,"# Password reset Password reset invalidates old sessions, issues a fresh token, and records t..."
/playbooks/billing/invoices.md,2.0000,"1, 3","# Invoice retries Billing retries failed card charges, regenerates invoice PDFs, and sends pa..."
/src/auth_service.py,4.0000,"1, 2, 8, 9","async def validate_token(token: str) -> bool: if token == ""expired"": return False return True..."
/src/billing_service.py,2.0000,"1, 2","def retry_invoice_charge(invoice_id: str) -> str: return f""retrying card charge for {invoice_..."


**lexical_search (4 hits)**

path,score,preview
/docs/postgres_backend.md,0.5000,"# Postgres backend PostgresFileSystem keeps grep, glob, lexical search, and pgvector similari..."
/src/auth_service.py,0.5000,"async def validate_token(token: str) -> bool: if token == ""expired"": return False return True..."
/playbooks/auth/login.md,0.2000,# Login flow Users sign in with email plus a one-time code. Session tokens expire after 15 mi...
/playbooks/auth/password-reset.md,0.1000,"# Password reset Password reset invalidates old sessions, issues a fresh token, and records t..."


In [7]:
query_text = "login session token authentication"
query_vector = toy_embed_text(query_text)

show_table(
    [{"axis": axis, "value": value} for axis, value in zip(AXES, query_vector)],
    title=f"Toy query vector for: {query_text!r}",
)

vector_result = await fs.vector_search(list(query_vector), k=4)
show_result(vector_result, columns=("path", "score"))

semantic_result = await fs.semantic_search("Which file explains the Postgres vector index?", k=4)
show_result(semantic_result, columns=("path", "score"))

top_hit = semantic_result.entries[0].path
top_hit_result = await fs.read(top_hit)
print(top_hit)
print()
print(top_hit_result.entries[0].content)

print("\nCLI pipeline:\n")
print(await fs.cli('glob "/src/**/*.py" | grep "token|charge"'))


**Toy query vector for: 'login session token authentication'**

axis,value
auth,1.0000
billing,0.0000
postgres,0.0000
python,0.0000


**vector_search (4 hits)**

path,score
/playbooks/auth/password-reset.md,1.0000
/playbooks/auth/login.md,0.9370
/src/auth_service.py,0.8321
/docs/postgres_backend.md,0.0000


**semantic_search (4 hits)**

path,score
/docs/postgres_backend.md,1.0000
/playbooks/auth/login.md,0.3123
/playbooks/auth/password-reset.md,0.0000
/playbooks/billing/invoices.md,0.0000


/docs/postgres_backend.md

# Postgres backend

PostgresFileSystem keeps grep, glob, lexical search, and pgvector similarity inside PostgreSQL. verify_native_search_schema checks the GIN full-text index and the ANN vector index before traffic hits the backend.


CLI pipeline:

/src/auth_service.py:1:async def validate_token(token: str) -> bool:
/src/auth_service.py:2:    if token == "expired":
/src/auth_service.py:8:    session_token = f"session:{email}"
/src/auth_service.py:9:    return session_token
/src/billing_service.py:1:def retry_invoice_charge(invoice_id: str) -> str:
/src/billing_service.py:2:    return f"retrying card charge for {invoice_id}"


## Try Your Own Queries

Ideas:

- Change `DEMO_FILES` and rerun the seeding cell.
- Change `query_text` to watch the vector ranking move around.
- Set `CONFIG.reset_table = False` if you want to keep your own writes between runs.
- Use `await fs.cli('...')` for pipelines once you know the direct Python calls.
- Inspect `pg_indexes` again after you change the schema.


In [ ]:
# A few handy experiments:
#
# write_result = await fs.write("/notes/ideas.md", "Postgres auth index notes")
# show_result(write_result, columns=("path", "kind", "size_bytes"))
#
# print(await fs.cli('glob "/**/*.md" | grep "postgres"'))
#
# Uncomment this to see the backend fail fast when the ANN index is missing:
# async with engine.begin() as conn:
#     await conn.execute(text(f"DROP INDEX IF EXISTS {vector_spec.index_name}"))
# await fs.verify_native_search_schema()
#
# When you're done:
# await engine.dispose()
